# Tutorial: Qdrant Internals Through Billboard Song Search

**Audience:** developers who know basic Python and want to understand what Qdrant does below its client API.

**Prerequisites:** Docker, Docker Compose, `uv`, Python 3.11, and the local Qdrant service from `deployment/compose.qdrant.yaml`.

**Learning goals:** follow writes through WAL and segments, compare exact scan with HNSW, combine similarity with filters, inspect parameters, mutate points, and verify persistence.

This notebook mirrors `docs/tutorials/qdrant_billboard_walkthrough.md`. It is educational and does not integrate with the FastAPI backend.

## Outline

Q00–Q04 establish prerequisites, data, and embeddings. Q05–Q10 create collections and compare exact/HNSW search. Q11–Q16 cover personal queries, filtering, parameters, mutation, persistence, and the final exercise.

In [ ]:
from __future__ import annotations

import math
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "starter.md").exists() and (candidate / "AGENTS.md").exists():
            return candidate
    raise RuntimeError("Launch Jupyter from inside the AskMyDocs repository")

REPO_ROOT = find_repo_root(Path.cwd())
SOURCE_ROOT = REPO_ROOT / "sandbox" / "qdrant_music" / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from qdrant_music.dataset import load_dataset, validation_summary
from qdrant_music.embedding import embed_documents
from qdrant_music.qdrant_ops import (
    collection_summary,
    create_payload_indexes,
    delete_collection,
    delete_point,
    ensure_collection,
    load_songs,
    search,
    update_review_note,
    wait_for_index,
    wait_until_ready,
)
from qdrant_music.settings import DEFAULT_QUERY, SETTINGS

pd.set_option("display.max_colwidth", 80)
REPO_ROOT

## Q00 — Learning goals and prerequisites

FastEmbed creates vectors locally; Qdrant validates, stores, indexes, filters, and searches them. Start the service before continuing:

```bash
docker compose -f deployment/compose.qdrant.yaml up -d
```

In [ ]:
try:
    readiness = wait_until_ready(timeout_seconds=5)
except TimeoutError:
    command = "docker compose -f deployment/compose.qdrant.yaml up -d"
    raise RuntimeError(f"Qdrant is unavailable. From the repository root run: {command}") from None
readiness

## Q01 — Qdrant's data model

```text
collection → shard → segments → vector storage + payload storage + indexes
```

A point is an external ID, one or more vectors, and optional JSON payload. Segments are independently searchable units. The WAL orders durable changes before update workers apply them to segments.

In [ ]:
# Cosine similarity in two dimensions before we use 768 dimensions.
a = (1.0, 1.0)
b = (2.0, 2.0)
dot = sum(x * y for x, y in zip(a, b))
cosine = dot / (math.sqrt(sum(x*x for x in a)) * math.sqrt(sum(y*y for y in b)))
{"dot_product": dot, "cosine_similarity": cosine}

The vectors have different magnitudes but identical direction, so cosine similarity is 1. Qdrant applies the same calculation to much larger vectors.

## Q02 — Connect and inspect the server

`/readyz` answers whether the node can serve requests. Port 6333 is REST; 6334 is gRPC. The Compose file binds both to loopback.

In [ ]:
readiness

## Q03 — Inspect and validate Billboard data

The chart facts are frozen; short original descriptions and manually reviewed genre/mood tags provide semantic text. Stable UUIDv5 point IDs make reruns idempotent.

In [ ]:
summary = validation_summary()
metadata, songs = load_dataset()
display(summary)
pd.DataFrame([
    {"rank": song.chart_rank, "title": song.title, "artist": song.artist, "tags": ", ".join(song.community_tags)}
    for song in songs[:10]
])

## Q04 — Generate and inspect an embedding

Nomic uses `search_document:` for stored descriptions and `search_query:` for queries. We show only shape, norm, and five values.

In [ ]:
first_vector = embed_documents([songs[0].description])[0]
{
    "song": songs[0].title,
    "dimensions": len(first_vector),
    "l2_norm": math.sqrt(sum(value * value for value in first_vector)),
    "first_five_values": first_vector[:5],
}

## Q05 — Create the baseline collection

The baseline uses 768 dimensions, cosine distance, and Qdrant defaults. Existing collections are verified, never silently deleted.

In [ ]:
baseline_name = ensure_collection("baseline", notebook=True)
baseline_name

## Q06 — Trace and execute an upsert

The client batch is validated, appended to the WAL, applied by an update worker to an appendable segment, and acknowledged with `wait=True`. Later optimizer work is asynchronous.

In [ ]:
baseline_load = load_songs("baseline", notebook=True)
baseline_load

## Q07 — Inspect segments and collection state

For 50 vectors, `indexed_vectors_count` may be zero because exact scanning is cheaper. Approximate operational counts can temporarily differ during optimization; use the exact count API for correctness.

In [ ]:
baseline_state = collection_summary("baseline", notebook=True)
pd.DataFrame([baseline_state]).T.rename(columns={0: "baseline"})

## Q08 — Run exact similarity search

Exact search compares the query with every eligible vector. It gives us ground truth for approximate-result overlap.

In [ ]:
exact_results = search(DEFAULT_QUERY, kind="baseline", notebook=True, exact=True)
pd.DataFrame(exact_results)[["score", "title", "artist", "chart_rank", "community_tags"]]

## Q09 — Build the HNSW learning collection

The learning collection lowers `indexing_threshold` and `full_scan_threshold` to 10 KiB, so this tiny dataset actually builds and uses HNSW. `m=16` controls graph connectivity; `ef_construct=100` controls build exploration.

In [ ]:
hnsw_load = load_songs("hnsw", notebook=True)
hnsw_state = wait_for_index(notebook=True, timeout_seconds=60)
pd.DataFrame([hnsw_state]).T.rename(columns={0: "hnsw"})

## Q10 — Compare exact and approximate search

Higher `hnsw_ef` explores more graph candidates and usually improves recall at a latency cost. With 50 points, overhead can make HNSW slower; this is not a capacity benchmark.

In [ ]:
comparison_rows = []
exact_hnsw_collection = search(DEFAULT_QUERY, kind="hnsw", notebook=True, exact=True)
exact_ids = {row["id"] for row in exact_hnsw_collection}
for label, exact, ef in [("exact", True, None), ("hnsw-8", False, 8), ("hnsw-32", False, 32), ("hnsw-128", False, 128)]:
    started = time.perf_counter()
    rows = search(DEFAULT_QUERY, kind="hnsw", notebook=True, exact=exact, hnsw_ef=ef)
    comparison_rows.append({
        "mode": label,
        "elapsed_ms": round((time.perf_counter() - started) * 1000, 3),
        "exact_top5_overlap": len(exact_ids & {row["id"] for row in rows}),
        "titles": ", ".join(row["title"] for row in rows),
    })
pd.DataFrame(comparison_rows)

## Q11 — Run a personal liking query

Edit these inputs. The query changes, while stored song vectors remain unchanged.

In [ ]:
PERSONAL_QUERY = "melancholic acoustic songs with calm intimate vocals"
RESULT_LIMIT = 5
SCORE_THRESHOLD = None
HNSW_EF = 64
USE_EXACT_SEARCH = False
RETURN_VECTORS = False

personal_results = search(
    PERSONAL_QUERY,
    kind="hnsw",
    notebook=True,
    limit=RESULT_LIMIT,
    score_threshold=SCORE_THRESHOLD,
    hnsw_ef=HNSW_EF,
    exact=USE_EXACT_SEARCH,
    with_vectors=RETURN_VECTORS,
)
pd.DataFrame(personal_results)[["score", "title", "artist", "community_tags", "description"]]

## Q12 — Add filters and payload indexes

Vector similarity answers “what feels related?” Payload filters enforce exact constraints. Typed indexes improve planning and traversal but do not change filter meaning.

In [ ]:
REQUIRED_TAG = "dance-pop"
MAX_CHART_RANK = 40
MIN_WEEKS_ON_CHART = None

started = time.perf_counter()
before_index = search(DEFAULT_QUERY, kind="hnsw", notebook=True, required_tag=REQUIRED_TAG, max_chart_rank=MAX_CHART_RANK, min_weeks_on_chart=MIN_WEEKS_ON_CHART)
before_ms = (time.perf_counter() - started) * 1000
create_payload_indexes("hnsw", notebook=True)
started = time.perf_counter()
after_index = search(DEFAULT_QUERY, kind="hnsw", notebook=True, required_tag=REQUIRED_TAG, max_chart_rank=MAX_CHART_RANK, min_weeks_on_chart=MIN_WEEKS_ON_CHART)
after_ms = (time.perf_counter() - started) * 1000
pd.DataFrame([
    {"stage": "before", "elapsed_ms": round(before_ms, 3), "titles": ", ".join(row["title"] for row in before_index)},
    {"stage": "after", "elapsed_ms": round(after_ms, 3), "titles": ", ".join(row["title"] for row in after_index)},
])

## Q13 — Explore parameter trade-offs

Record whether each control affects recall, latency, result volume, memory, storage, or correctness. The full catalog is in the Markdown tutorial.

In [ ]:
parameter_rows = [
    {"parameter": "m", "phase": "build", "higher_value": "more graph links; often more recall, RAM, and build time", "rebuild": True},
    {"parameter": "ef_construct", "phase": "build", "higher_value": "better graph construction; slower build", "rebuild": True},
    {"parameter": "hnsw_ef", "phase": "query", "higher_value": "more candidates; often better recall and slower query", "rebuild": False},
    {"parameter": "score_threshold", "phase": "query", "higher_value": "fewer, more similar results", "rebuild": False},
    {"parameter": "limit", "phase": "query", "higher_value": "more returned candidates and work", "rebuild": False},
    {"parameter": "on_disk", "phase": "storage", "higher_value": "memory mapping lowers required RAM; adds I/O sensitivity", "rebuild": True},
]
pd.DataFrame(parameter_rows)

## Q14 — Update and delete points

A logical delete stops search visibility immediately, while old physical bytes may remain until vacuum optimization. The stable-ID upsert restores the point.

In [ ]:
target = songs[0]
update_review_note(target.point_id, "Updated from notebook Q14", kind="baseline", notebook=True)
count_after_delete = delete_point(target.point_id, kind="baseline", notebook=True)
count_after_restore = load_songs("baseline", notebook=True)["exact_count"]
{"point_id": target.point_id, "after_delete": count_after_delete, "after_restore": count_after_restore}

## Q15 — Verify persistence

For a full persistence exercise, set `RUN_PERSISTENCE_RESTART=True`. Normal run-all leaves Docker untouched and verifies current durable state. Compose `down` preserves the named volume; `down --volumes` destroys it.

In [ ]:
RUN_PERSISTENCE_RESTART = False
compose_file = REPO_ROOT / "deployment" / "compose.qdrant.yaml"
if RUN_PERSISTENCE_RESTART:
    subprocess.run(["docker", "compose", "-f", str(compose_file), "down"], cwd=REPO_ROOT, check=True)
    subprocess.run(["docker", "compose", "-f", str(compose_file), "up", "-d"], cwd=REPO_ROOT, check=True)
    wait_until_ready(timeout_seconds=30)
persistence_state = collection_summary("baseline", notebook=True)
assert persistence_state["exact_count"] == 50
persistence_state

## Q16 — Final exercise and cleanup

Change the query and one filter. Explain the top three using tags and descriptions. Cleanup is optional. Docker-volume deletion is deliberately never executed here.

In [ ]:
FINAL_QUERY = "confident high-energy music for a road trip"
final_results = search(FINAL_QUERY, kind="hnsw", notebook=True, hnsw_ef=64, max_chart_rank=50, limit=5)
pd.DataFrame(final_results)[["score", "title", "artist", "community_tags", "description"]]

In [ ]:
CLEANUP_NOTEBOOK_COLLECTIONS = False
CONFIRM_DESTRUCTIVE_RESET = False  # Deletes every collection in the sandbox volume when True.

if CLEANUP_NOTEBOOK_COLLECTIONS:
    cleanup = {
        "baseline_deleted": delete_collection("baseline", notebook=True),
        "hnsw_deleted": delete_collection("hnsw", notebook=True),
    }
else:
    cleanup = {"collections_retained": True}

if CONFIRM_DESTRUCTIVE_RESET:
    subprocess.run(
        ["docker", "compose", "-f", str(compose_file), "down", "--volumes"],
        cwd=REPO_ROOT,
        check=True,
    )
    cleanup = {"destructive_volume_reset": True}
cleanup

## Exercise answer scaffold

- My query expressed these mood/style/context signals: …
- I selected exact/HNSW because: …
- My filter guaranteed: …
- The top three matched because their tags/descriptions contained: …
- Changing `hnsw_ef` changed exact overlap from … to …

**Common pitfall:** a similarity score is not a probability. Interpret it only relative to the model, metric, query, and candidate set.

**Extension:** after the approved hybrid-retrieval epic, compare these dense results with sparse/BM25 candidates rather than adding that product feature here.